In [0]:
%run "../includes/configuration"

In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment", "dev", "environment")
v_env = dbutils.widgets.get("p_environment")

In [0]:
from pyspark.sql.functions import col, lit, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

In [0]:
language_role_df = (
    spark.read
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/language_role.json")
    )
display(language_role_df)

In [0]:
language_role_schema = StructType([
    StructField("languageRole", StringType()),
    StructField("roleId", IntegerType())
])

language_role_df = (
    spark.read
        .schema(language_role_schema)
        .format("json")
        .option("multiline", "true")
        .load(f"{bronze_folder_path}/language_role.json")
)
display(language_role_df)

In [0]:
language_role_df = (
    language_role_df
        .withColumnsRenamed(
            {
                "languageRole": "language_role",
                "roleId": "role_id"
            }

        )
        .withColumns(
            {
                "env": lit(v_env)
            }
        )
)

language_role_df = add_ingestion_date(language_role_df)

display(language_role_df)

In [0]:
(
    language_role_df
        .write
        .format("parquet")
        .mode("overwrite")
        .save(f"{silver_folder_path}/language_roles")
)

In [0]:
dbutils.notebook.exit("Success")